# Circulation data
Join all the files in the reanalysis, reduce to scope (10S to 10N, 180W to 20E), output single ds with U, W

In [5]:
import xarray as xr
import numpy as np
import os
from dask_jobqueue import PBSCluster
from dask.distributed import Client

In [6]:
cluster = PBSCluster(
    cores=1, # The number of cores you want
    memory='128GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    resource_spec='select=1:ncpus=1:mem=128GB', # Specify resources
    account='P93300313', # Input your project ID here
    walltime='01:00:00', # Amount of wall time
    n_workers=10,
)
# Setup your client
client = Client(cluster)

/glade/u/home/acruz/.conda/envs/EOFa_2025/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35451 instead
  warnings.warn(


In [7]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/35451/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/35451/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.184:35857,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/35451/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Import and select data

## Select U names

In [8]:
# dates in scope 
hist_decades = ['191001-191912', '192001-192912',
                '193001-193912', '194001-194912', '195001-195912', '196001-196912',
                '197001-197912', '198001-198912', '199001-199912', '200001-200912', 
                '201001-201412']

# too many different member names (macro and micro perturbations)
macro_name = np.char.mod('%d', np.arange(1001, 1311, 10)).tolist()

micro_name = np.char.zfill(np.char.mod('%d', np.arange(1, 21, 1)), 3).tolist()


path = "/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/"
title = "b.e21.BHIST"
type = '.nc'

all_Us = []
for a in ["cmip6", "smbb"]:
    hist = f'{title}{a}'
    for i in macro_name:
        for j in micro_name:
            member = []
            for z in hist_decades:
                file = path + hist + '.f09_g17.LE2-'+ i + '.' +  j + '.cam.h0.U.' + z + type
                # print(file)
                if os.path.exists(file):
                    member.append(file)
                else:
                    continue
            if member != []:
                all.append(member)

In [9]:
all_ds = []
for m in all:
    member = []
    for f in m:
        print(f)
        ds = xr.open_dataset(f, engine='h5netcdf', chunks={})['U']
        member.append(ds)
    member_data = xr.concat(member, dim='time')
    all_ds.append(member_data)

/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.191001-191912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.192001-192912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.193001-193912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.194001-194912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.195001-195912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.196001-196912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.197001-197912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm

In [10]:
Uds = xr.concat(all_ds, dim='member')
Uds

<xarray.DataArray 'U' (member: 200, time: 1260, lev: 32, lat: 192, lon: 288)> Size: 2TB
dask.array<concatenate, shape=(200, 1260, 32, 192, 288), dtype=float32, chunksize=(1, 1, 16, 96, 144), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
Dimensions without coordinates: member
Attributes:
    mdims:         1
    units:         m/s
    long_name:     Zonal wind
    cell_methods:  time: mean

# Sort coords

In [18]:
ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180
ds = ds.sortby(ds.longitude)

/glade/u/home/acruz/.local/lib/python3.10/site-packages/xarray/core/indexing.py:1680: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


In [19]:
# making sure the latitude transformation works properly
newlats = (ds.coords['latitude'] + 90) % 180 - 90
newlats.sortby(newlats.latitude)

<xarray.DataArray 'latitude' (latitude: 721)> Size: 6kB
array([-90.  , -89.75, -89.5 , -89.25, -89.  , -88.75, -88.5 , -88.25,
       -88.  , -87.75, -87.5 , -87.25, -87.  , -86.75, -86.5 , -86.25,
       -86.  , -85.75, -85.5 , -85.25, -85.  , -84.75, -84.5 , -84.25,
       -84.  , -83.75, -83.5 , -83.25, -83.  , -82.75, -82.5 , -82.25,
       -82.  , -81.75, -81.5 , -81.25, -81.  , -80.75, -80.5 , -80.25,
       -80.  , -79.75, -79.5 , -79.25, -79.  , -78.75, -78.5 , -78.25,
       -78.  , -77.75, -77.5 , -77.25, -77.  , -76.75, -76.5 , -76.25,
       -76.  , -75.75, -75.5 , -75.25, -75.  , -74.75, -74.5 , -74.25,
       -74.  , -73.75, -73.5 , -73.25, -73.  , -72.75, -72.5 , -72.25,
       -72.  , -71.75, -71.5 , -71.25, -71.  , -70.75, -70.5 , -70.25,
       -70.  , -69.75, -69.5 , -69.25, -69.  , -68.75, -68.5 , -68.25,
       -68.  , -67.75, -67.5 , -67.25, -67.  , -66.75, -66.5 , -66.25,
       -66.  , -65.75, -65.5 , -65.25, -65.  , -64.75, -64.5 , -64.25,
       -64.  , -63.75, -63.5 , -63.25, -63.  , -62.75, -62.5 , -62.25,
       -62.  , -61.75, -61.5 , -61.25, -61.  , -60.75, -60.5 , -60.25,
       -60.  , -59.75, -59.5 , -59.25, -59.  , -58.75, -58.5 , -58.25,
       -58.  , -57.75, -57.5 , -57.25, -57.  , -56.75, -56.5 , -56.25,
       -56.  , -55.75, -55.5 , -55.25, -55.  , -54.75, -54.5 , -54.25,
       -54.  , -53.75, -53.5 , -53.25, -53.  , -52.75, -52.5 , -52.25,
       -52.  , -51.75, -51.5 , -51.25, -51.  , -50.75, -50.5 , -50.25,
...
        52.  ,  52.25,  52.5 ,  52.75,  53.  ,  53.25,  53.5 ,  53.75,
        54.  ,  54.25,  54.5 ,  54.75,  55.  ,  55.25,  55.5 ,  55.75,
        56.  ,  56.25,  56.5 ,  56.75,  57.  ,  57.25,  57.5 ,  57.75,
        58.  ,  58.25,  58.5 ,  58.75,  59.  ,  59.25,  59.5 ,  59.75,
        60.  ,  60.25,  60.5 ,  60.75,  61.  ,  61.25,  61.5 ,  61.75,
        62.  ,  62.25,  62.5 ,  62.75,  63.  ,  63.25,  63.5 ,  63.75,
        64.  ,  64.25,  64.5 ,  64.75,  65.  ,  65.25,  65.5 ,  65.75,
        66.  ,  66.25,  66.5 ,  66.75,  67.  ,  67.25,  67.5 ,  67.75,
        68.  ,  68.25,  68.5 ,  68.75,  69.  ,  69.25,  69.5 ,  69.75,
        70.  ,  70.25,  70.5 ,  70.75,  71.  ,  71.25,  71.5 ,  71.75,
        72.  ,  72.25,  72.5 ,  72.75,  73.  ,  73.25,  73.5 ,  73.75,
        74.  ,  74.25,  74.5 ,  74.75,  75.  ,  75.25,  75.5 ,  75.75,
        76.  ,  76.25,  76.5 ,  76.75,  77.  ,  77.25,  77.5 ,  77.75,
        78.  ,  78.25,  78.5 ,  78.75,  79.  ,  79.25,  79.5 ,  79.75,
        80.  ,  80.25,  80.5 ,  80.75,  81.  ,  81.25,  81.5 ,  81.75,
        82.  ,  82.25,  82.5 ,  82.75,  83.  ,  83.25,  83.5 ,  83.75,
        84.  ,  84.25,  84.5 ,  84.75,  85.  ,  85.25,  85.5 ,  85.75,
        86.  ,  86.25,  86.5 ,  86.75,  87.  ,  87.25,  87.5 ,  87.75,
        88.  ,  88.25,  88.5 ,  88.75,  89.  ,  89.25,  89.5 ,  89.75,
       -90.  ])
Coordinates:
  * latitude  (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0

In [20]:
ds = ds.sortby(newlats.latitude)
ds

<xarray.Dataset> Size: 162GB
Dimensions:    (time: 528, level: 37, latitude: 721, longitude: 1440)
Coordinates:
  * latitude   (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * time       (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2022-12-01
  * longitude  (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
Data variables:
    U          (time, level, latitude, longitude) float32 81GB dask.array<chunksize=(96, 37, 721, 1440), meta=np.ndarray>
    utc_date   (time) int32 2kB dask.array<chunksize=(96,), meta=np.ndarray>
    W          (time, level, latitude, longitude) float32 81GB dask.array<chunksize=(96, 37, 721, 1440), meta=np.ndarray>
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.6.1
    CONVERSION_PLATFORM:  Linux casper04 3.10.0-693.21.1.el7.x86_64 #1 SMP We...
    CONVERSION_DATE:      Fri Nov  8 18:03:42 MST 2019
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Fri Nov  8 18:03:52 2019: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 4.7.9 (Homepage = http://n...

# Select Data

In [ ]:
tropic = ds.sel(latitude=slice(-10, 10), longitude=slice(-180, 20))
tropic.compute()
tropic

In [ ]:
tropic.to_netcdf('/glade/work/acruz/Reanalysis/ERA5_circulation.nc')